In [ ]:
#@title Cell 53.1 — Aim and fixed clinical interpretation
from IPython.display import Markdown, display

display(Markdown(r"""
# NB53 — BMC-N resistance-threshold coverage across all 26 antibiotics (v0.1)

**Data:** The independently held-out 50,460 BMC-N predictions from Notebook 41. Observed MICs alone determine whether an isolate is above the chosen EUCAST resistance cutoff. No Model C or BMC-N outputs are used to select the subgroup.

**Breakpoint source:** EUCAST Clinical Breakpoint Tables v16.1 (24 June–31 December 2026), **Enterobacterales / *E. coli***, https://www.eucast.org/fileadmin/eucast/pdf/breakpoints/v_16.1_Breakpoint_Tables.pdf (printed pages 15–20).

**IMPORTANT:** Infection indication and specimen source are unavailable in the dataset. Some thresholds apply ONLY to urinary infections or other indications. Results for those rows are **threshold-defined subsets**, not verified clinical resistance classifications for every patient. Systemic aminoglycoside numbers shown *in parentheses* by EUCAST are mechanism-oriented cutoffs and are **not** used as general clinical breakpoints. Agents with screen-only, notes-only, or no clinical breakpoint are excluded with reasons.

Outputs: all-26 audit, main one-row-per-applicable-antibiotic table, alternate-indication sensitivity table, full resistance-group IDs and provenance metadata. The main table reports **inside, above and below** the BMC-N 95% interval, with counts and percentages.
"""))


In [ ]:
#@title Cell 53.2 — Locate the validated Notebook 41 input and define output paths
import os
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
except ImportError:
    pass

ROOT = Path(os.environ.get('BMC_PROJECT_ROOT', '/content/drive/MyDrive/RefMIC'))
if not ROOT.is_dir():
    raise FileNotFoundError(f'RefMIC root not found: {ROOT}. Set BMC_PROJECT_ROOT for local Jupyter.')

INPUT_NAME = '41_BMC_N_independent_outer_test_predictions.csv.gz'
EXPECTED = ROOT / '03_Results' / '41_BMC_N_Pathogen_Out_Validation' / INPUT_NAME
OUT = ROOT / '03_Results' / '53_BMC_Resistance_Threshold_Coverage'
OUT.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for part in iter(lambda: fh.read(4 * 1024 * 1024), b''):
            h.update(part)
    return h.hexdigest()

def locate_nb41():
    if EXPECTED.is_file():
        return EXPECTED
    matches = []
    for dirname, dirs, names in os.walk(ROOT):
        dirs[:] = [d for d in dirs if d not in {'.git', '__pycache__', 'node_modules'}]
        if INPUT_NAME in names:
            matches.append(Path(dirname) / INPUT_NAME)
    if not matches:
        raise FileNotFoundError(f'Missing Notebook 41 output {INPUT_NAME}. Expected: {EXPECTED}. Run Cell 41.8 first.')
    hashes = {sha256_file(p) for p in matches}
    if len(hashes) > 1:
        raise ValueError('Different Notebook 41 predictions found. Keep authoritative file at ' + str(EXPECTED))
    return sorted(matches, key=lambda p: (len(p.parts), str(p)))[0]

PRED_PATH = locate_nb41()
print('Input:', PRED_PATH)
print('Output:', OUT)


In [ ]:
#@title Cell 53.3 — Pre-specified EUCAST v16.1 cutoffs and exclusion audit
# Values are EUCAST MIC resistance criteria: resistant when observed MIC > R_cutoff_mg_L.
# Only a single stated primary scenario per antibiotic is displayed in the main table.
# Indication-dependent thresholds have explicit restrictions; not generalized to all patients.
# EUCAST source: v16.1, Enterobacterales table, printed pp. 15–20.

EUCAST_URL = 'https://www.eucast.org/fileadmin/eucast/pdf/breakpoints/v_16.1_Breakpoint_Tables.pdf'
# (antibiotic, R>, indication/scenario, limitation or interpretive note, EUCAST printed page)
PRIMARY = [
    ('amikacin', 8.0, 'Infections originating from urinary tract', 'Urinary-origin breakpoint; systemic value (8) is bracketed.', 18),
    ('amoxicillin-clavulanic acid', 8.0, 'Intravenous / infections other than uncomplicated UTI', 'MIC interpreted as amoxicillin concentration with clavulanate fixed at 2 mg/L; oral uncomplicated UTI cutoff differs (32).', 15),
    ('ampicillin', 8.0, 'Intravenous / general indication', 'EUCAST Enterobacterales ampicillin iv.', 15),
    ('cefazolin', 4.0, 'Infections originating from urinary tract', 'Urinary-origin E. coli criterion; no unrestricted cefazolin threshold.', 16),
    ('cefepime', 4.0, 'General indication', 'EUCAST Enterobacterales.', 16),
    ('cefotaxime', 2.0, 'Indications other than meningitis', 'Meningitis resistance threshold differs (1 mg/L).', 16),
    ('ceftazidime', 4.0, 'General indication', 'EUCAST Enterobacterales.', 16),
    ('ceftriaxone', 2.0, 'Indications other than meningitis', 'Meningitis resistance threshold differs (1 mg/L).', 16),
    ('cephalexin', 16.0, 'Uncomplicated urinary tract infection only', 'EUCAST row spells cefalexin.', 16),
    ('ciprofloxacin', 0.5, 'Indications other than meningitis', 'Meningitis criterion differs (0.125 mg/L); ATU at MIC 0.5 mg/L.', 17),
    ('gentamicin', 2.0, 'Infections originating from urinary tract', 'Urinary-origin breakpoint; systemic value (2) is bracketed.', 18),
    ('imipenem', 4.0, 'Enterobacterales except Morganellaceae', 'Applies to E. coli.', 17),
    ('meropenem', 8.0, 'Indications other than meningitis', 'Meningitis resistance criterion differs (2 mg/L).', 17),
    ('piperacillin-tazobactam', 8.0, 'General indication', 'Tazobactam fixed at 4 mg/L; EUCAST ATU at MIC 16 mg/L.', 15),
    ('temocillin', 16.0, 'Infections originating from urinary tract, E. coli', 'Not an unrestricted Enterobacterales clinical threshold.', 15),
    ('tigecycline', 0.5, 'E. coli', 'Species-specific EUCAST clinical breakpoint.', 19),
    ('tobramycin', 2.0, 'Infections originating from urinary tract', 'Urinary-origin breakpoint; systemic value (2) is bracketed.', 18),
    ('trimethoprim-sulfamethoxazole', 0.5, 'Enterobacterales except Serratia', 'MIC must denote the trimethoprim component of the 1:19 combination. Verify MIC units in original data.', 20),
]
EXCLUDED = {
    'azithromycin': 'Note only for Enterobacterales; no clinical MIC resistance breakpoint.',
    'cefoxitin': 'Screen only for AmpC; no clinical MIC resistance breakpoint.',
    'chloramphenicol': 'Screening cutoff for acquired resistance only; no clinical MIC resistance breakpoint.',
    'doxycycline': 'No E. coli clinical MIC resistance breakpoint.',
    'nalidixic acid': 'Screen only; no clinical MIC resistance breakpoint.',
    'streptomycin': 'No Enterobacterales clinical MIC resistance breakpoint.',
    'sulfisoxazole': 'No Enterobacterales clinical MIC resistance breakpoint for this single agent.',
    'tetracycline': 'No E. coli clinical MIC resistance breakpoint.',
}
ALTERNATIVES = [
    ('amoxicillin-clavulanic acid', 32.0, 'Oral, uncomplicated UTI', 'Alternative R criterion for oral uncomplicated UTI.', 15),
    ('cefotaxime', 1.0, 'Meningitis', 'Alternative indication-specific R threshold.', 16),
    ('ceftriaxone', 1.0, 'Meningitis', 'Alternative indication-specific R threshold.', 16),
    ('ciprofloxacin', 0.125, 'Meningitis', 'Alternative indication-specific R threshold.', 17),
    ('meropenem', 2.0, 'Meningitis', 'Alternative indication-specific R threshold.', 17),
]
PRIMARY_DF = pd.DataFrame(PRIMARY, columns=['antibiotic','r_cutoff_mg_L_exclusive','clinical_scenario','interpretive_note','eucast_printed_page'])
ALTERNATIVE_DF = pd.DataFrame(ALTERNATIVES, columns=['antibiotic','r_cutoff_mg_L_exclusive','clinical_scenario','interpretive_note','eucast_printed_page'])
assert len(PRIMARY_DF) == 18 and len(EXCLUDED) == 8 and PRIMARY_DF.antibiotic.is_unique
assert not (set(PRIMARY_DF.antibiotic) & set(EXCLUDED))
print('Applicable primary R criteria:',len(PRIMARY_DF),' | No clinical MIC R breakpoint:',len(EXCLUDED))
display(PRIMARY_DF[['antibiotic','r_cutoff_mg_L_exclusive','clinical_scenario','interpretive_note']])


In [ ]:
#@title Cell 53.4 — Load and validate all 50,460 independent BMC-N test observations
bmc = pd.read_csv(PRED_PATH, compression='gzip')
required = {
    'interaction_row', 'biosample', 'antibiotic', 'outer_fold',
    'observed_log2_mic', 'model_c_outer_test_predicted_log2_mic',
    'bmc_n_lower95_log2','bmc_n_upper95_log2'
}
missing = sorted(required - set(bmc.columns))
if missing:
    raise ValueError('Notebook 41 output is missing columns: ' + ', '.join(missing))

bmc['antibiotic'] = bmc.antibiotic.astype(str).str.lower().str.strip()
assert (len(bmc), bmc.biosample.nunique(), bmc.antibiotic.nunique()) == (50460, 9058, 26), 'Wrong held-out cohort dimensions.'
assert set(bmc.antibiotic) == (set(PRIMARY_DF.antibiotic) | set(EXCLUDED)), 'Breakpoint table misses an antibiotic from Notebook 41.'
assert set(bmc.outer_fold) == set(range(1,6)), 'Expected five test folds.'
assert not bmc.interaction_row.duplicated().any(), 'Duplicate interaction rows.'
assert not bmc.duplicated(['biosample','antibiotic']).any(), 'Duplicate pathogen–antibiotic observation.'
assert (bmc.groupby('biosample').outer_fold.nunique() == 1).all(), 'A pathogen belongs to more than one outer fold.'
vals=['observed_log2_mic','model_c_outer_test_predicted_log2_mic','bmc_n_lower95_log2','bmc_n_upper95_log2']
assert np.isfinite(bmc[vals].to_numpy(dtype=float)).all(), 'Nonfinite test data.'
assert (bmc.bmc_n_upper95_log2 > bmc.bmc_n_lower95_log2).all(), 'Invalid prediction limits.'

bmc['bmc_n_within_95pct'] = ((bmc.observed_log2_mic >= bmc.bmc_n_lower95_log2) &
                            (bmc.observed_log2_mic <= bmc.bmc_n_upper95_log2))
bmc['bmc_n_above_upper95'] = bmc.observed_log2_mic > bmc.bmc_n_upper95_log2
bmc['bmc_n_below_lower95'] = bmc.observed_log2_mic < bmc.bmc_n_lower95_log2
bmc['bmc_n_interval_width_log2'] = bmc.bmc_n_upper95_log2 - bmc.bmc_n_lower95_log2
assert (bmc[['bmc_n_within_95pct','bmc_n_above_upper95','bmc_n_below_lower95']].sum(axis=1) == 1).all(), 'Interval status is not mutually exclusive and exhaustive.'
print('Validated:',len(bmc),'held-out MICs,',bmc.biosample.nunique(),'BioSamples,',bmc.antibiotic.nunique(),'antibiotics.')


In [ ]:
#@title Cell 53.5 — Compute all-antibiotic and alternate-indication resistance-group coverage

def evaluate_threshold(row):
    antibiotic = row.antibiotic
    cutoff = float(row.r_cutoff_mg_L_exclusive)
    subset = bmc.loc[bmc.antibiotic.eq(antibiotic)].copy()
    # Compare on log2 scale without rounding the original MIC values.
    res = subset.loc[subset.observed_log2_mic > np.log2(cutoff)].copy()
    n_all, n_r = len(subset), len(res)
    n_in = int(res.bmc_n_within_95pct.sum())
    n_upper = int(res.bmc_n_above_upper95.sum())
    n_lower = int(res.bmc_n_below_lower95.sum())
    assert n_in + n_upper + n_lower == n_r
    return dict(
        antibiotic=antibiotic,
        r_cutoff_mg_L_exclusive=cutoff,
        clinical_scenario=row.clinical_scenario,
        n_tested=n_all,
        n_above_r_cutoff=n_r,
        pct_tested_above_cutoff=100*n_r/n_all,
        n_within95=n_in,
        pct_within95_resistant=100*n_in/n_r if n_r else np.nan,
        n_outside95=n_r-n_in,
        pct_outside95_resistant=100*(n_r-n_in)/n_r if n_r else np.nan,
        n_above_upper95=n_upper,
        pct_above_upper95_resistant=100*n_upper/n_r if n_r else np.nan,
        n_below_lower95=n_lower,
        pct_below_lower95_resistant=100*n_lower/n_r if n_r else np.nan,
        pct_within95_all=100*subset.bmc_n_within_95pct.mean(),
        delta_resistant_minus_all_coverage_pp=(100*n_in/n_r -100*subset.bmc_n_within_95pct.mean()) if n_r else np.nan,
        mean_interval_width_log2_resistant=res.bmc_n_interval_width_log2.mean() if n_r else np.nan,
        n_at_mic16_in_resistant=int(np.isclose(res.observed_log2_mic.to_numpy(float),4.0,atol=1e-9).sum()) if antibiotic=='piperacillin-tazobactam' else np.nan,
        interpretive_note=row.interpretive_note,
        eucast_printed_page=row.eucast_printed_page,
        eucast_reference=EUCAST_URL
    )

primary_coverage = pd.DataFrame([evaluate_threshold(row) for row in PRIMARY_DF.itertuples(index=False)])
alternative_coverage = pd.DataFrame([evaluate_threshold(row) for row in ALTERNATIVE_DF.itertuples(index=False)])

# Reconcile NB52's previously verified resistant-group sizes and exact BMC-N coverage.
for abx, nr, n_in in [('cefepime',449,421),('gentamicin',416,308)]:
    row=primary_coverage.loc[primary_coverage.antibiotic.eq(abx)].iloc[0]
    if (row.n_above_r_cutoff, row.n_within95) != (nr,n_in):
        raise ValueError(f'NB53 does not reproduce audited NB52 {abx} counts: got {(row.n_above_r_cutoff,row.n_within95)}; expected {(nr,n_in)}')

excluded = pd.DataFrame([{
    'antibiotic': antibiotic,
    'n_tested': int(bmc.antibiotic.eq(antibiotic).sum()),
    'clinical_r_cutoff_status': 'Not applicable',
    'exclusion_reason': reason,
    'eucast_reference': EUCAST_URL
} for antibiotic,reason in sorted(EXCLUDED.items())])

print('Clinical resistance-cutoff subsets (the indication caveats in the table apply):')
display(primary_coverage[['antibiotic','r_cutoff_mg_L_exclusive','n_tested','n_above_r_cutoff',
                          'n_within95','pct_within95_resistant','n_outside95','pct_outside95_resistant',
                          'n_above_upper95','n_below_lower95']].round(2))
print('Excluded from EUCAST clinical-R analysis:')
display(excluded)


In [ ]:
#@title Cell 53.6 — Supplementary results table, ranking and selection cautions
# Rank only antibiotics with at least 30 threshold-defined R observations.
# Coverage ranking of small strata is unstable, hence display them separately without a rank.
primary_coverage['coverage_rank_if_n_r_ge30'] = np.nan
eligible = primary_coverage[primary_coverage.n_above_r_cutoff >= 30].sort_values('pct_within95_resistant')
primary_coverage.loc[eligible.index,'coverage_rank_if_n_r_ge30'] = np.arange(1,len(eligible)+1)

supp_table = primary_coverage.sort_values(['pct_within95_resistant','antibiotic'],na_position='last').copy()
print('Supplementary table (sorted by threshold-defined resistant-group coverage):')
display(supp_table[['antibiotic','clinical_scenario','r_cutoff_mg_L_exclusive',
                    'n_tested','n_above_r_cutoff','n_within95','pct_within95_resistant',
                    'n_outside95','pct_outside95_resistant','n_above_upper95','n_below_lower95',
                    'mean_interval_width_log2_resistant','coverage_rank_if_n_r_ge30']].round(2))
print('Number of antibiotics with >=30 threshold-defined resistant observations:',len(eligible))
print('The ranking is descriptive, depends on the EUCAST indication and sample size, and does not establish clinical validity for unknown infection indications.')
print('Check TMP-SMX MIC coding before claiming clinical resistance; do not interpret bracketed aminoglycoside systemic values as generic clinical breakpoints.')


In [ ]:
#@title Cell 53.7 — Export reproducible Supplementary Table inputs, audit and provenance
import json

# Include all 26 antibiotics, retaining exclusions transparently.
all_26 = PRIMARY_DF[['antibiotic','r_cutoff_mg_L_exclusive','clinical_scenario','interpretive_note']].copy()
all_26['clinical_r_cutoff_status'] = 'Applicable under stated scenario'
excl_all = excluded[['antibiotic','n_tested','exclusion_reason']].copy()
excl_all['r_cutoff_mg_L_exclusive'] = np.nan
excl_all['clinical_scenario'] = 'No valid E. coli clinical MIC R breakpoint'
excl_all['interpretive_note'] = excl_all.exclusion_reason
excl_all['clinical_r_cutoff_status'] = 'Not applicable'

main_cols = ['antibiotic','r_cutoff_mg_L_exclusive','clinical_scenario','interpretive_note','clinical_r_cutoff_status']
all_26_audit = pd.concat([all_26[main_cols],excl_all[main_cols]],ignore_index=True)
all_26_audit = all_26_audit.merge(primary_coverage.drop(columns=['r_cutoff_mg_L_exclusive','clinical_scenario','interpretive_note']),on='antibiotic',how='left')
all_26_audit['n_tested'] = all_26_audit.antibiotic.map(bmc.antibiotic.value_counts()).astype(int)
all_26_audit = all_26_audit.sort_values('antibiotic').reset_index(drop=True)
assert len(all_26_audit)==26 and all_26_audit.n_tested.sum()==50460

# Auditable identities for all subsets; does not replace source NB41 file.
full_subsets = []
for row in PRIMARY_DF.itertuples(index=False):
    r = bmc.loc[(bmc.antibiotic.eq(row.antibiotic)) &
                (bmc.observed_log2_mic > np.log2(row.r_cutoff_mg_L_exclusive))].copy()
    r['r_cutoff_mg_L_exclusive'] = row.r_cutoff_mg_L_exclusive
    r['clinical_scenario'] = row.clinical_scenario
    full_subsets.append(r)
full_subsets = pd.concat(full_subsets, ignore_index=True)

files = {
 '53_Supplementary_Resistance_Coverage_18_Antibiotics.csv':supp_table,
 '53_All_26_Antibiotic_Breakpoint_And_Exclusion_Audit.csv':all_26_audit,
 '53_Alternative_Indication_Sensitivity.csv':alternative_coverage,
 '53_Full_Resistance_Threshold_Defined_Observations.csv.gz':full_subsets,
}
for filename,table in files.items():
    path=OUT/filename
    table.to_csv(path,index=False,compression='gzip' if filename.endswith('.gz') else None)
    print('Saved',path,'(',len(table),'rows)')
metadata = {
  'notebook':'NB53 v0.1',
  'source_nb41_path':str(PRED_PATH),
  'source_nb41_sha256':sha256_file(PRED_PATH),
  'eucast_version':'16.1, valid from 2026-06-24',
  'eucast_source':EUCAST_URL,
  'cohort_observations':len(bmc),
  'unique_biosamples':int(bmc.biosample.nunique()),
  'antibiotics':int(bmc.antibiotic.nunique()),
  'applicable_primary_clinical_scenarios':len(PRIMARY_DF),
  'excluded_no_clinical_mic_breakpoint':len(EXCLUDED),
  'selection_rule':'observed_log2_mic > log2(EUCAST R cutoff mg/L)',
  'interval_coverage_rule':'lower95 <= observed_log2_mic <= upper95',
  'qualification':'Unknown clinical indication: urinary-only and other indication-specific rows are MIC-threshold-defined, not confirmed all-indication clinical resistance.',
  'other_qualification':'TMP-SMX requires source MIC scale to be trimethoprim component; verify ratio before interpreting clinically.'
}
with open(OUT/'53_Provenance_And_Analysis_Definitions.json','w') as fh:
    json.dump(metadata,fh,indent=2)
print('Saved provenance. Audit complete.')
